# S3 Tables as Federated Glue Catalog on EMR Serverless via SparkConnect and Iceberg Features



<a id="prerequisites"></a>
## Prerequisites
<div class="alert alert-block alert-info">
<b>NOTE :</b> In order to execute this notebook successfully as is, please ensure the following prerequisites are completed.</div>

**What This Scenario Covers:**

This notebook demonstrates how to use **Apache Spark Connect** on **Amazon EMR Serverless** to run PySpark from a local Jupyter environment. EMR Serverless handles driver and executor provisioning,runs the DataFrame and SQL operations on your local notebook sends operations and receives results.

On top of the Spark Connect setup, this notebook goes further: it registers **S3 Tables as a federated catalog in AWS Glue**, creates **Apache Iceberg tables**, loads supply chain sample data, runs multi-table analytical queries, and exercises core Iceberg capabilities including time travel and schema evolution.

| Field | Value |
|-------|-------|
| Environment | Local Jupyter → EMR Serverless (Spark Connect) |
| EMR Release | emr-7.13.0 (Spark 3.5.6) |
| Region | us-west-2 |

**What you'll build and test:**

| # | What's Being Tested |
|---|---------------------|
| 1 | Local PySpark client ↔ remote EMR Serverless session via Spark Connect |
| 2 | S3 Tables registered as a federated Glue catalog |
| 3 | Iceberg table creation and data loading via Spark SQL |
| 4 | Multi-table analytical queries (supply chain, warehouse, low stock) |
| 5 | Iceberg snapshots, time travel, and schema evolution |

**Prerequisites:**
* AWS credentials with permissions for: `emr-serverless:StartSession`, `GetSession`, `GetSessionEndpoint`, `TerminateSession`, `GetResourceDashboard`, and `iam:PassRole` on the runtime role
* An EMR Serverless application (`emr-7.13.0` or later) with `interactiveConfiguration.sessionEnabled = true`
* IAM execution role with permissions for S3Tables, S3, Glue, and CloudWatch
* S3 bucket in `us-west-2` for EMR Serverless logs
* `boto3 >= 1.35.0`
* Python 3.9+ with pip
* AWS CLI 
* Familiarity with PySpark (Level 200)

**Documentation:** [Amazon EMR Serverless Spark Connect](https://docs.aws.amazon.com/emr/latest/EMR-Serverless-UserGuide/spark-connect.html)

***

## Introduction

Apache Iceberg (https://iceberg.apache.org/) is an open table format for huge analytic datasets. Iceberg adds tables to compute engines including Spark, Trino, PrestoDB, Flink and Hive using a high-performance table format that works just like a SQL table.

In this notebook we use **S3 Tables** as a federated catalog registered in the **AWS Glue Data Catalog**, accessed via **EMR Serverless SparkConnect** from a local Jupyter environment. We demonstrate:

- Creating Iceberg tables on S3 Tables
- Loading sample data (products, inventory, shipments)
- Running analytical queries (supply chain, warehouse performance, low stock alerts)
- Iceberg features: **Snapshots**, **Time Travel**, and **Schema Evolution**

***

<a id="install_deps"></a>
## Step 1: Install Local Dependencies

Run these in your terminal before launching the notebook:

```bash
pip install pyspark==3.5.6
pip install pandas numpy pyarrow grpcio grpcio-status googleapis-common-protos
pip install --upgrade boto3 botocore
```

<div class="alert alert-block alert-warning">
⚠️ PySpark version must match the EMR Spark version exactly. Mismatch causes gRPC errors.<br>
⚠️ boto3 must be 1.35.0 or higher for s3tables client support.
</div>

Verify:

In [ ]:
import pyspark
print(f"PySpark version: {pyspark.__version__}")
# Expected: 3.5.6

import boto3
print(f"boto3 version: {boto3.__version__}")
# Expected: 1.35.0 or higher

<a id="configure_cli"></a>
## Step 2: Configure AWS CLI 

Set these environment variables in your terminal:

```bash
export APP_ID=<YOUR_APP_ID>
export ROLE_ARN=arn:aws:iam::<YOUR_ACCOUNT_ID>:role/EMRServerlessS3RuntimeRole
export EP=https://emr-serverless.us-west-2.amazonaws.com/
export REGION=us-west-2
```

Verify session commands are available:

```bash
aws emr-serverless help | grep session
# Expected:
#   o get-session
#   o get-session-endpoint
#   o list-sessions
#   o start-session
#   o terminate-session
```


<a id="iam_role"></a>
## Step 3: Set Up IAM Execution Role

Create a trust policy `trust-policy.json`:

```json
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": { "Service": "emr-serverless.amazonaws.com" },
            "Action": "sts:AssumeRole"
        }
    ]
}
```

Apply it:

```bash
aws iam update-assume-role-policy \
    --role-name EMRServerlessS3RuntimeRole \
    --policy-document file://trust-policy.json
```

**Attach AWS Managed Policies:**

```bash
aws iam attach-role-policy --role-name EMRServerlessS3RuntimeRole \
    --policy-arn arn:aws:iam::aws:policy/AmazonAthenaFullAccess
aws iam attach-role-policy --role-name EMRServerlessS3RuntimeRole \
    --policy-arn arn:aws:iam::aws:policy/AmazonS3TablesFullAccess
aws iam attach-role-policy --role-name EMRServerlessS3RuntimeRole \
    --policy-arn arn:aws:iam::aws:policy/AWSGlueConsoleFullAccess
```

| Policy Name | Type | Purpose |
|---|---|---|
| AmazonAthenaFullAccess | AWS managed | Athena query access |
| AmazonS3TablesFullAccess | AWS managed | Full S3 Tables access |
| AWSGlueConsoleFullAccess | AWS managed | Glue console + API access |
| EMRServerlessS3LogsAccess | Customer inline | EMR logs bucket access |
| EMRServerlessS3AndGlueAccessPolicy | Customer managed | Glue catalog + S3 Tables + Lake Formation |


<a id="register_catalog"></a>
## Step 4: Register S3 Tables as a Federated Glue Catalog

<div class="alert alert-block alert-warning">
⚠️ Do this before creating the EMR application so the catalog is visible when Spark initializes.
</div>

Create `catalog.json`:

```json
{
    "Name": "s3tablescatalog",
    "CatalogInput": {
        "Description": "Federated catalog for all S3 table buckets in this account and region",
        "FederatedCatalog": {
            "Identifier": "arn:aws:s3tables:us-west-2:<YOUR_ACCOUNT_ID>:bucket/*",
            "ConnectionName": "aws:s3tables"
        },
        "CreateDatabaseDefaultPermissions": [{
            "Principal": { "DataLakePrincipalIdentifier": "IAM_ALLOWED_PRINCIPALS" },
            "Permissions": ["ALL"]
        }],
        "CreateTableDefaultPermissions": [{
            "Principal": { "DataLakePrincipalIdentifier": "IAM_ALLOWED_PRINCIPALS" },
            "Permissions": ["ALL"]
        }],
        "AllowFullTableExternalDataAccess": "True"
    }
}
```

```bash
aws glue create-catalog \
    --name "s3tablescatalog" \
    --cli-input-json file://catalog.json \
    --region ${REGION}

# Verify
aws glue get-catalog \
    --catalog-id s3tablescatalog \
    --region ${REGION}
```

Expected: catalog with `FederatedCatalog.ConnectionName = "aws:s3tables"`.


<a id="create_app"></a>
## Step 5: Create & Start EMR Serverless Application

<div class="alert alert-block alert-warning">
⚠️ All Iceberg and catalog configs MUST be set at the application level via --runtime-configuration. Session-level configuration-overrides are silently ignored.<br>
⚠️ <code>cache-enabled: false</code> is required.<br>
⚠️ <code>glue.id</code> must include the bucket name suffix: <code>&lt;ACCOUNT_ID&gt;:s3tablescatalog/&lt;BUCKET_NAME&gt;</code>
</div>

```bash
aws emr-serverless create-application \
    --name "s3tables-sparkconnect-lakehouse" \
    --type SPARK \
    --release-label emr-7.13.0 \
    --initial-capacity '{
        "Driver":   { "workerCount": 2, "workerConfiguration": {"cpu": "4vcpu", "memory": "16gb", "disk": "20GB"} },
        "Executor": { "workerCount": 4, "workerConfiguration": {"cpu": "4vcpu", "memory": "16gb", "disk": "20GB"} }
    }' \
    --runtime-configuration '[{
        "classification": "spark-defaults",
        "properties": {
            "spark.sql.catalog.s3tablescatalog":               "org.apache.iceberg.spark.SparkCatalog",
            "spark.sql.catalog.s3tablescatalog.type":          "glue",
            "spark.sql.catalog.s3tablescatalog.glue.id":       "<YOUR_ACCOUNT_ID>:s3tablescatalog/<BUCKET_NAME>",
            "spark.sql.defaultCatalog":                        "s3tablescatalog",
            "spark.sql.extensions":                            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
            "spark.sql.catalog.s3tablescatalog.cache-enabled": "false"
        }
    }]' \
    --monitoring-configuration '{
        "managedPersistenceMonitoringConfiguration": {"enabled": true},
        "s3MonitoringConfiguration": {"logUri": "s3://YOUR-BUCKET/logs/sparkconnect-"}
    }' \
    --interactive-configuration '{"sessionEnabled": true}' \
    --endpoint-url ${EP} \
    --region ${REGION}
```

```bash
export APP_ID=<applicationId from response>

# Start the application
aws emr-serverless start-application \
    --application-id ${APP_ID} \
    --endpoint-url ${EP} \
    --region ${REGION}

# Wait for STARTED (~90 seconds)
aws emr-serverless get-application \
    --application-id ${APP_ID} \
    --endpoint-url ${EP} \
    --region ${REGION} | grep state
# Expected: "state": "STARTED"
```


<a id="start_session"></a>
## Step 6: Start EMR Serverless Session

```bash
aws emr-serverless start-session \
    --application-id ${APP_ID} \
    --execution-role-arn ${ROLE_ARN} \
    --name "s3tables-spark-session" \
    --idle-timeout-minutes 60 \
    --endpoint-url ${EP} \
    --region ${REGION}
```

```bash
export SESSION_ID=<sessionId from response>
```


## Step 7: Wait for Session IDLE State

```bash
aws emr-serverless list-sessions \
    --application-id ${APP_ID} \
    --endpoint-url ${EP} \
    --region ${REGION}
```

Expected: `"state": "IDLE"`, `"stateDetails": "Session is running."`


<a id="get_endpoint"></a>
## Step 8: Get SparkConnect Endpoint & Auth Token

```bash
aws emr-serverless get-session-endpoint \
    --application-id ${APP_ID} \
    --session-id ${SESSION_ID} \
    --endpoint-url https://emr-serverless.us-west-2.amazonaws.com/ \
    --region us-west-2
```

```bash
export SPARK_CONNECT_ENDPOINT=<endpoint from response>
export AUTH_TOKEN=<authToken from response>
```


***

<a id="connect_spark"></a>
## Step 9: Connect from Local Jupyter via SparkConnect

<div class="alert alert-block alert-warning">
⚠️ The warehouse ARN <b>MUST</b> be set via <code>.config()</code> on SparkSession.builder <b>before</b> <code>.getOrCreate()</code>. Setting it via <code>spark.conf.set()</code> after connection fails with <i>Stream removed</i>.
</div>


In [ ]:
from pyspark.sql import SparkSession

# Stop any existing session first
try:
    spark.stop()
    print("✅ Old session stopped")
except:
    pass

SPARK_CONNECT_ENDPOINT = "<YOUR_ENDPOINT>"
AUTH_TOKEN             = "<YOUR_AUTH_TOKEN>"
WAREHOUSE_ARN          = "arn:aws:s3tables:us-west-2:<YOUR_ACCOUNT_ID>:bucket/<YOUR_BUCKET_NAME>"

url = (
    f"sc://{SPARK_CONNECT_ENDPOINT}:443/;"
    f"use_ssl=true;"
    f"x-aws-proxy-auth={AUTH_TOKEN}"
)

spark = (
    SparkSession.builder
    .remote(url)
    .config("spark.sql.catalog.s3tablescatalog.warehouse", WAREHOUSE_ARN)
    .getOrCreate()
)

print(f"✅ Connected! Spark version: {spark.version}")
print(f"✅ Warehouse: {spark.conf.get('spark.sql.catalog.s3tablescatalog.warehouse', 'NOT SET')}")
spark.sql("SELECT 1 + 1 AS result").show()

<a id="verify_catalog"></a>
## Step 10: Verify Catalog Configuration

Confirm all Spark/Iceberg catalog settings are active on the remote session.


In [ ]:
for key in [
    "spark.sql.catalog.s3tablescatalog",
    "spark.sql.catalog.s3tablescatalog.type",
    "spark.sql.catalog.s3tablescatalog.glue.id",
    "spark.sql.catalog.s3tablescatalog.warehouse",
    "spark.sql.catalog.s3tablescatalog.cache-enabled",
    "spark.sql.defaultCatalog",
    "spark.sql.extensions",
]:
    try:
        val = spark.conf.get(key)
        print(f"✅ {key} = {val}")
    except Exception:
        print(f"❌ {key} = NOT SET")


<a id="explore_catalog"></a>
## Step 11: Explore Catalogs, Namespaces & Tables

Let's browse the existing catalog structure to see what namespaces and tables are already available.


### Show Namespaces


In [ ]:
spark.sql("SHOW NAMESPACES").show()


### Show Catalogs


In [ ]:
spark.sql("SHOW CATALOGS").show()

### Show Existing Tables


In [ ]:
spark.sql("SHOW TABLES IN s3tablescatalog.s3table_db").show(truncate=False)
spark.sql("SHOW TABLES IN s3tablescatalog.sttable").show(truncate=False)


### Set Active Catalog and Namespace


In [ ]:
spark.sql("USE s3tablescatalog")
spark.sql("USE s3tablescatalog.sttable")
print("✅ Catalog and namespace set!")


In [ ]:
spark.sql("SELECT current_catalog(), current_database()").show(truncate=False)


In [ ]:
spark.sql("SHOW TABLES IN s3tablescatalog.sttable").show(truncate=False)

In [ ]:
spark.sql("SHOW TABLES IN s3tablescatalog.sttable").toPandas()

In [ ]:
spark.sql("SELECT current_catalog(), current_database()").show(truncate=False)

***

<a id="create_tables"></a>
## Step 12: Create Iceberg Tables

We create three tables — `products`, `inventory`, and `shipments` — using `USING iceberg` syntax. These tables are stored in S3 Tables and managed via the federated Glue catalog.

<div class="alert alert-block alert-warning">
⚠️ Requires warehouse to be set at connection time (Step 9).
</div>


In [ ]:
for table, ddl in [
    ("products", """
        CREATE TABLE IF NOT EXISTS s3tablescatalog.sttable.products (
            product_id  INT,
            name        STRING,
            category    STRING,
            unit_price  DOUBLE,
            supplier    STRING
        ) USING iceberg
    """),
    ("inventory", """
        CREATE TABLE IF NOT EXISTS s3tablescatalog.sttable.inventory (
            inventory_id INT,
            product_id   INT,
            warehouse    STRING,
            quantity     INT,
            last_updated DATE
        ) USING iceberg
    """),
    ("shipments", """
        CREATE TABLE IF NOT EXISTS s3tablescatalog.sttable.shipments (
            shipment_id INT,
            product_id  INT,
            destination STRING,
            quantity    INT,
            ship_date   DATE,
            status      STRING
        ) USING iceberg
    """),
]:
    try:
        spark.sql(ddl)
        print(f"✅ {table} created!")
    except Exception as e:
        print(f"❌ {table} failed: {e}")

In [ ]:
spark.sql("SHOW TABLES IN s3tablescatalog.sttable").show(truncate=False)


<a id="load_data"></a>
## Step 13: Load Data & Run Analytics


<a id="load_products"></a>
### a) Load Products


In [ ]:
import random
from datetime import date

random.seed(42)

# ---- products ----
categories = ["electronics", "apparel", "home", "sports"]
suppliers  = ["supplier_a", "supplier_b", "supplier_c"]

p_rows = ", ".join([
    f"({i}, 'product_{i}', '{random.choice(categories)}', "
    f"{round(random.uniform(10, 999), 2)}, '{random.choice(suppliers)}')"
    for i in range(1, 21)
])

spark.sql(f"INSERT INTO s3tablescatalog.sttable.products VALUES {p_rows}")
print(f"✅ Products loaded: {spark.sql('SELECT count(*) FROM s3tablescatalog.sttable.products').collect()[0][0]} rows")

In [ ]:
# Preview
spark.sql("SELECT * FROM s3tablescatalog.sttable.products LIMIT 5").show(truncate=False)


<a id="load_inventory"></a>
### b) Load Inventory


In [ ]:
# ---- inventory ----
warehouses = ["seattle", "portland", "sf"]

i_rows = ", ".join([
    f"({i}, {random.randint(1, 20)}, '{random.choice(warehouses)}', "
    f"{random.randint(0, 500)}, DATE'2025-04-{random.randint(1,15):02d}')"
    for i in range(1, 31)
])

spark.sql(f"INSERT INTO s3tablescatalog.sttable.inventory VALUES {i_rows}")
print(f"✅ Inventory loaded: {spark.sql('SELECT count(*) FROM s3tablescatalog.sttable.inventory').collect()[0][0]} rows")

# Preview
spark.sql("SELECT * FROM s3tablescatalog.sttable.inventory LIMIT 5").show(truncate=False)


<a id="load_shipments"></a>
### c) Load Shipments


In [ ]:
# ---- shipments ----
destinations = ["new_york", "chicago", "dallas", "miami"]
statuses     = ["delivered", "delivered", "in_transit", "pending"]

s_rows = ", ".join([
    f"({i}, {random.randint(1, 20)}, '{random.choice(destinations)}', "
    f"{random.randint(1, 50)}, DATE'2025-0{random.randint(1,4)}-{random.randint(1,28):02d}', "
    f"'{random.choice(statuses)}')"
    for i in range(1, 51)
])

spark.sql(f"INSERT INTO s3tablescatalog.sttable.shipments VALUES {s_rows}")
print(f"✅ Shipments loaded: {spark.sql('SELECT count(*) FROM s3tablescatalog.sttable.shipments').collect()[0][0]} rows")

# Preview
spark.sql("SELECT * FROM s3tablescatalog.sttable.shipments LIMIT 5").show(truncate=False)


### Row Count Summary


In [ ]:
print("📊 Row Count Summary:")
print("=" * 40)
for table in ["products", "inventory", "shipments"]:
    count = spark.sql(f"SELECT count(*) FROM s3tablescatalog.sttable.{table}").collect()[0][0]
    print(f"  ✅ {table:12} : {count} rows")
print("=" * 40)

<a id="supply_chain"></a>
### d) Supply Chain Analysis

Full analysis showing product flow from inventory to shipment.


In [ ]:
print("📊 Full Supply Chain Analysis - Product Flow from Inventory to Shipment:")
spark.sql("""
    SELECT 
        p.category,
        p.name                                        AS product_name,
        p.supplier,
        p.unit_price,
        SUM(i.quantity)                               AS total_inventory,
        COUNT(DISTINCT i.warehouse)                   AS warehouses_stocked,
        COUNT(s.shipment_id)                          AS total_shipments,
        SUM(s.quantity)                               AS total_units_shipped,
        ROUND(SUM(s.quantity * p.unit_price), 2)      AS total_revenue,
        ROUND(SUM(i.quantity * p.unit_price), 2)      AS inventory_value,
        ROUND(SUM(s.quantity) * 100.0 / 
              SUM(i.quantity), 2)                     AS sell_through_rate_pct
    FROM s3tablescatalog.sttable.products   p
    LEFT JOIN s3tablescatalog.sttable.inventory  i ON p.product_id = i.product_id
    LEFT JOIN s3tablescatalog.sttable.shipments  s ON p.product_id = s.product_id
                                                   AND s.status = 'delivered'
    GROUP BY p.category, p.name, p.supplier, p.unit_price
    ORDER BY total_revenue DESC
    LIMIT 10
""").show(truncate=False)


<a id="warehouse_perf"></a>
### e) Warehouse Performance

Stock value vs shipped value and utilization percentage across all warehouses.


In [ ]:
print("📊 Warehouse Performance - Stock Value vs Shipped Value:")
spark.sql("""
    SELECT 
        i.warehouse,
        COUNT(DISTINCT i.product_id)                  AS unique_products,
        SUM(i.quantity)                               AS current_stock,
        ROUND(SUM(i.quantity * p.unit_price), 2)      AS stock_value,
        COUNT(DISTINCT s.shipment_id)                 AS shipments_out,
        SUM(s.quantity)                               AS units_shipped,
        ROUND(SUM(s.quantity * p.unit_price), 2)      AS shipped_value,
        ROUND(SUM(s.quantity * p.unit_price) * 100.0 /
              SUM(i.quantity * p.unit_price), 2)      AS utilization_pct
    FROM s3tablescatalog.sttable.inventory   i
    JOIN s3tablescatalog.sttable.products    p ON i.product_id  = p.product_id
    LEFT JOIN s3tablescatalog.sttable.shipments s ON i.product_id = s.product_id
                                                  AND s.status = 'delivered'
    GROUP BY i.warehouse
    ORDER BY stock_value DESC
""").show(truncate=False)


<a id="low_stock"></a>
### f) Low Stock Alert

Products with high shipment demand and low available stock.


In [ ]:
print("📊 Low Stock Alert - Products with High Shipment Demand:")
spark.sql("""
    SELECT
        p.product_id,
        p.name                                        AS product_name,
        p.category,
        p.supplier,
        p.unit_price,
        i.warehouse,
        i.quantity                                    AS current_stock,
        COUNT(s.shipment_id)                          AS pending_shipments,
        SUM(CASE WHEN s.status IN ('pending','in_transit') 
                 THEN s.quantity ELSE 0 END)          AS units_committed,
        i.quantity - SUM(CASE WHEN s.status IN 
                              ('pending','in_transit') 
                              THEN s.quantity 
                              ELSE 0 END)             AS available_stock,
        CASE 
            WHEN i.quantity = 0 THEN '🔴 OUT OF STOCK'
            WHEN i.quantity < 50 THEN '🟡 LOW STOCK'
            ELSE '🟢 OK'
        END                                           AS stock_status
    FROM s3tablescatalog.sttable.products    p
    JOIN s3tablescatalog.sttable.inventory   i ON p.product_id = i.product_id
    LEFT JOIN s3tablescatalog.sttable.shipments s ON p.product_id = s.product_id
    GROUP BY p.product_id, p.name, p.category, 
             p.supplier, p.unit_price, 
             i.warehouse, i.quantity
    ORDER BY available_stock ASC
    LIMIT 10
""").show(truncate=False)


***

<a id="iceberg_features"></a>
## Step 14: Iceberg Features


<a id="snapshots"></a>
### a) View Current Snapshots

Every DML operation creates a new snapshot. Let's see the current state.


In [ ]:
# View current snapshots for all tables
print("📸 Current Snapshots - shipments:")
spark.sql("""
    SELECT snapshot_id, committed_at, operation 
    FROM s3tablescatalog.sttable.shipments.snapshots
""").show(truncate=False)

print("📸 Current Snapshots - products:")
spark.sql("""
    SELECT snapshot_id, committed_at, operation 
    FROM s3tablescatalog.sttable.products.snapshots
""").show(truncate=False)

print("📸 Current Snapshots - inventory:")
spark.sql("""
    SELECT snapshot_id, committed_at, operation 
    FROM s3tablescatalog.sttable.inventory.snapshots
""").show(truncate=False)


<a id="time_travel"></a>
### b) Time Travel — Insert New Row & Compare Snapshots

Iceberg's time travel lets you query historical data at any previous snapshot. Let's insert a new row, then compare the old and new snapshots.


In [ ]:
# Get current row count before insert
before = spark.sql("""
    SELECT count(*) FROM s3tablescatalog.sttable.shipments
""").collect()[0][0]
print(f"📊 Rows before insert: {before}")

# Insert new row — creates a new snapshot
spark.sql("""
    INSERT INTO s3tablescatalog.sttable.shipments VALUES 
    (51, 5, 'london', 25, DATE'2025-05-01', 'delivered')
""")
print("✅ New row inserted!")

# Get snapshots ordered by time
snaps = spark.sql("""
    SELECT snapshot_id, committed_at, operation 
    FROM s3tablescatalog.sttable.shipments.snapshots 
    ORDER BY committed_at
""").collect()

print(f"\n📸 Total snapshots: {len(snaps)}")
for s in snaps:
    print(f"  snapshot_id: {s[0]} | committed_at: {s[1]} | operation: {s[2]}")

# Query old snapshot vs current
old_snapshot_id = snaps[0][0]
old_count = spark.sql(f"""
    SELECT count(*) 
    FROM s3tablescatalog.sttable.shipments 
    VERSION AS OF {old_snapshot_id}
""").collect()[0][0]

current_count = spark.sql("""
    SELECT count(*) FROM s3tablescatalog.sttable.shipments
""").collect()[0][0]

print(f"\n✅ Time Travel Results:")
print(f"   Snapshot 1 (old) : {old_count} rows")
print(f"   Current          : {current_count} rows")

#### Query Data at Specific Snapshot


In [ ]:
old_snapshot_id = snaps[0][0]

# Show data at old snapshot
print(f"📊 Data at old snapshot ({old_count} rows):")
spark.sql(f"""
    SELECT * 
    FROM s3tablescatalog.sttable.shipments 
    VERSION AS OF {old_snapshot_id}
    ORDER BY shipment_id DESC 
    LIMIT 5
""").show(truncate=False)

# Show current data
print(f"📊 Current data ({current_count} rows):")
spark.sql("""
    SELECT * 
    FROM s3tablescatalog.sttable.shipments 
    ORDER BY shipment_id DESC 
    LIMIT 5
""").show(truncate=False)


<a id="schema_evolution"></a>
### c) Schema Evolution

Iceberg tracks columns by unique IDs, not names. You can safely add, drop, rename, or reorder columns without rewriting data. Schema evolution is a metadata-only operation.


In [ ]:
# Check schema before
print("📋 Schema BEFORE evolution:")
spark.sql("""
    DESCRIBE TABLE s3tablescatalog.sttable.shipments
""").show(truncate=False)

# Add new column
spark.sql("""
    ALTER TABLE s3tablescatalog.sttable.shipments 
    ADD COLUMNS (discount_pct DOUBLE)
""")
print("✅ New column discount_pct added!")

# Check schema after
print("\n📋 Schema AFTER evolution:")
spark.sql("""
    DESCRIBE TABLE s3tablescatalog.sttable.shipments
""").show(truncate=False)

#### Verify NULL Backfill & Insert New Row with New Column

Existing rows show `NULL` for the new column — no data rewrite needed. New rows can populate the new column.


In [ ]:
print("📊 Old rows show NULL for new column - no data rewrite needed:")
spark.sql("""
    SELECT shipment_id,
           destination,
           quantity,
           status,
           discount_pct,
           CASE
               WHEN discount_pct IS NULL THEN '⚠️ Legacy row - no discount'
               ELSE '✅ New row - has discount'
           END AS row_type
    FROM s3tablescatalog.sttable.shipments
    ORDER BY shipment_id DESC
    LIMIT 5
""").show(truncate=False)

# Insert new row with discount_pct populated
spark.sql("""
    INSERT INTO s3tablescatalog.sttable.shipments VALUES 
    (52, 3, 'paris', 10, DATE'2025-05-02', 'pending', 15.0)
""")
print("✅ New row with discount_pct=15.0 inserted!")

# Show mix of NULL and non-NULL
print("\n📊 Mix of legacy and new rows:")
spark.sql("""
    SELECT shipment_id,
           destination,
           status,
           discount_pct,
           CASE
               WHEN discount_pct IS NULL THEN '⚠️ Legacy row'
               ELSE '✅ New row'
           END AS row_type
    FROM s3tablescatalog.sttable.shipments
    ORDER BY shipment_id DESC
    LIMIT 5
""").show(truncate=False)


***

<a id="spark_ui"></a>
## Step 15: Get Spark UI / Resource Dashboard

Two-step process required:

**Step 1 — Run in Terminal to Get URL:**
```bash
aws emr-serverless get-resource-dashboard \
    --application-id ${APP_ID} \
    --resource-id ${SESSION_ID} \
    --resource-type SESSION \
    --endpoint-url ${EP} \
    --region ${REGION}
```

Copy the `url` value from the response.

**Step 2 — Paste URL in Jupyter and Open:**


In [ ]:
import webbrowser

# Paste URL from terminal output here
url = "<paste_url_from_terminal_output>"

print("✅ Opening Spark UI in browser...")
print(f"\n🔗 Session  : <YOUR_SESSION_ID>")
print(f"🔗 App ID   : <YOUR_APP_ID>")
print(f"🔗 Region   : us-west-2")
webbrowser.open(url)


***

<a id="cleanup"></a>
## Step 16: Clean Up

### From the Notebook — Drop Tables and Stop Session


In [ ]:
# Drop tables
spark.sql("DROP TABLE IF EXISTS s3tablescatalog.sttable.shipments")
spark.sql("DROP TABLE IF EXISTS s3tablescatalog.sttable.inventory")
spark.sql("DROP TABLE IF EXISTS s3tablescatalog.sttable.products")

spark.stop()
print("✅ Spark session stopped!")

### From the Terminal — Terminate Session and Stop Application

```bash
# Terminate session
aws emr-serverless terminate-session \
    --application-id ${APP_ID} \
    --session-id ${SESSION_ID} \
    --endpoint-url ${EP} \
    --region ${REGION}

# Stop application
aws emr-serverless stop-application \
    --application-id ${APP_ID} \
    --endpoint-url ${EP} \
    --region ${REGION}
```

### Optional — Delete Table Bucket and Glue Catalog

```bash
export TABLE_BUCKET_ARN=arn:aws:s3tables:${REGION}:<YOUR_ACCOUNT_ID>:bucket/<YOUR_BUCKET_NAME>

aws s3tables delete-namespace \
    --table-bucket-arn ${TABLE_BUCKET_ARN} \
    --namespace sttable \
    --region ${REGION}

aws s3tables delete-table-bucket \
    --table-bucket-arn ${TABLE_BUCKET_ARN} \
    --region ${REGION}

aws glue delete-catalog \
    --catalog-id s3tablescatalog \
    --region ${REGION}
```
